In [16]:
import pandas as pd
import numpy as np
from scipy.stats import zscore

# Dataset simulado
dados = {
    'id_transacao': range(1, 21),
    'pagador': ['A', 'B', 'A', 'C', 'B', 'A', 'D', 'C', 'A', 'E', 'F', 'B', 'A', 'D', 'C', 'E', 'A', 'B', 'D', 'F'],
    'recebedor': ['X', 'Y', 'X', 'Z', 'Y', 'Z', 'Y', 'X', 'X', 'Z', 'X', 'Y', 'X', 'Z', 'X', 'Y', 'Z', 'Y', 'X', 'Z'],
    'valor': [100, 200, 150, 300, 200, 120, 100, 400, 100, 500, 600, 200, 150, 300, 400, 100, 150, 250, 350, 300],
    'data': pd.date_range('2024-01-01', periods=20, freq='D')
}
df = pd.DataFrame(dados)

# Função para calcular as métricas
def calcular_metricas(df):
    df['transacoes_unicas_pagador_recebedor'] = 0
    df['recebedores_unicos_por_pagador'] = 0
    df['pagadores_unicos_por_recebedor'] = 0
    df['razao_valor_frequencia'] = 0
    df['primeira_transacao_pagador'] = False
    df['primeira_transacao_recebedor'] = False
    df['zscore_valor'] = 0.0
    df['concentracao_valores'] = 0

    # Iteração transação por transação
    for i in range(len(df)):
        sub_df = df.iloc[:i + 1]  # Considerar somente transações até a atual

        # Número de transações únicas entre o mesmo pagador e recebedor
        contagem_pagador_recebedor = sub_df.groupby(['pagador', 'recebedor']).size()
        df.loc[i, 'transacoes_unicas_pagador_recebedor'] = contagem_pagador_recebedor.get(
            (df.loc[i, 'pagador'], df.loc[i, 'recebedor']), 0
        )

        # Número de recebedores únicos por pagador
        recebedores_unicos = sub_df.groupby('pagador')['recebedor'].nunique()
        df.loc[i, 'recebedores_unicos_por_pagador'] = recebedores_unicos.get(df.loc[i, 'pagador'], 0)

        # Número de pagadores únicos por recebedor
        pagadores_unicos = sub_df.groupby('recebedor')['pagador'].nunique()
        df.loc[i, 'pagadores_unicos_por_recebedor'] = pagadores_unicos.get(df.loc[i, 'recebedor'], 0)

        # Razão valor/frequência para o pagador
        grupo_pagador = sub_df[sub_df['pagador'] == df.loc[i, 'pagador']]
        total_valor = grupo_pagador['valor'].sum()
        total_transacoes = len(grupo_pagador)
        df.loc[i, 'razao_valor_frequencia'] = total_valor / total_transacoes if total_transacoes > 0 else 0

        # Primeira transação do pagador/recebedor
        df.loc[i, 'primeira_transacao_pagador'] = df.loc[i, 'pagador'] not in sub_df['pagador'][:-1].values
        df.loc[i, 'primeira_transacao_recebedor'] = df.loc[i, 'recebedor'] not in sub_df['recebedor'][:-1].values

        # Z-Score do valor por pagador
        valores_pagador = sub_df[sub_df['pagador'] == df.loc[i, 'pagador']]['valor']
        if len(valores_pagador) > 1:
            zscores = zscore(valores_pagador, nan_policy='omit')
            df.loc[i, 'zscore_valor'] = zscores.iloc[-1] if not np.isnan(zscores).all() else 0
        else:
            df.loc[i, 'zscore_valor'] = 0

        # Concentração de valores por pagador
        contagem_valores_pagador = sub_df[sub_df['pagador'] == df.loc[i, 'pagador']].groupby('valor').size()
        df.loc[i, 'concentracao_valores'] = contagem_valores_pagador.max() if not contagem_valores_pagador.empty else 0

    return df

# Aplicar função para calcular as métricas
df = calcular_metricas(df)

# Visualizar DataFrame com as novas métricas
df


C:\Users\pedro\AppData\Local\Temp\ipykernel_19072\3907875534.py:48: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '123.33333333333333' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[i, 'razao_valor_frequencia'] = total_valor / total_transacoes if total_transacoes > 0 else 0


,id_transacao,pagador,recebedor,valor,data,transacoes_unicas_pagador_recebedor,recebedores_unicos_por_pagador,pagadores_unicos_por_recebedor,razao_valor_frequencia,primeira_transacao_pagador,primeira_transacao_recebedor,zscore_valor,concentracao_valores
0,1,A,X,100,2024-01-01,1,1,1,100.000000,True,True,0.000000,1
1,2,B,Y,200,2024-01-02,1,1,1,200.000000,True,True,0.000000,1
2,3,A,X,150,2024-01-03,2,1,1,125.000000,False,False,1.000000,1
3,4,C,Z,300,2024-01-04,1,1,1,300.000000,True,True,0.000000,1
4,5,B,Y,200,2024-01-05,2,1,1,200.000000,False,False,0.000000,2
5,6,A,Z,120,2024-01-06,1,2,2,123.333333,False,False,-0.162221,1
6,7,D,Y,100,2024-01-07,1,1,2,100.000000,True,False,0.000000,1
7,8,C,X,400,2024-01-08,1,2,2,350.000000,False,False,1.000000,1
8,9,A,X,100,2024-01-09,3,2,2,117.500000,False,False,-0.855186,2
9,10,E,Z,500,2024-01-10,1,1,3,500.000000,True,False,0.000000,1
